In [1]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms as transforms

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device!")

# standard imports
import matplotlib.pyplot as plt
import numpy as np
import random

# sklearn data
from sklearn.datasets import make_blobs
from sklearn.datasets import make_circles
from sklearn.datasets import make_classification
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# sklearn models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# sklearn metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay


from torch import nn
from torch.utils.data import DataLoader

Using cpu device!


In [2]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# download test data from open datasets
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [3]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
len(classes)

10

In [4]:
len(training_data)

60000

In [5]:
X, y = next(iter(training_data))
print(X.shape)

torch.Size([1, 28, 28])


https://machinelearningmastery.com/building-a-logistic-regression-classifier-in-pytorch/

website used to help make this model


In [6]:
images, _ = next(iter(training_data))
mean = images.mean()
std = images.std()
print (mean, std)

tensor(0.3814) tensor(0.3994)


In [17]:
trans = transforms.Compose([transforms.ToTensor(), transforms.Normalize(0.5, 0.5)])
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform = trans
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform= trans
)

batch_size = 32
train_loader = DataLoader(training_data, batch_size = batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size = batch_size, shuffle=False)

In [18]:
class LogisticRegression(torch.nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        self.linear = torch.nn.Linear(n_inputs, n_outputs)

    def forward(self, x):
        x = x.view(x.size(0), -1)   
        return self.linear(x)

In [19]:
n_inputs = 28 * 28
n_outputs = 10 # len(classes)

model = LogisticRegression(n_inputs, n_outputs)

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.03,
    momentum=0.9,
    weight_decay=1e-4
)

epochs = 80



In [21]:
Loss = []
Acc = []

for epoch in range(epochs):

    
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)  
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    Loss.append(loss.item())

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.view(-1, 28*28)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print("Test Accuracy:", acc)


Test Accuracy: 0.8107
Test Accuracy: 0.799
Test Accuracy: 0.8101
Test Accuracy: 0.7882
Test Accuracy: 0.809
Test Accuracy: 0.7995
Test Accuracy: 0.8243
Test Accuracy: 0.7877
Test Accuracy: 0.7967
Test Accuracy: 0.8038
Test Accuracy: 0.7414
Test Accuracy: 0.7758
Test Accuracy: 0.8056
Test Accuracy: 0.7869
Test Accuracy: 0.8218
Test Accuracy: 0.7979
Test Accuracy: 0.7983
Test Accuracy: 0.8062
Test Accuracy: 0.7785
Test Accuracy: 0.8167
Test Accuracy: 0.7449
Test Accuracy: 0.7815
Test Accuracy: 0.7927
Test Accuracy: 0.7984
Test Accuracy: 0.8046
Test Accuracy: 0.7989
Test Accuracy: 0.8042
Test Accuracy: 0.8179
Test Accuracy: 0.7966
Test Accuracy: 0.8136
Test Accuracy: 0.8189
Test Accuracy: 0.7914
Test Accuracy: 0.7963
Test Accuracy: 0.8063
Test Accuracy: 0.8228
Test Accuracy: 0.8264
Test Accuracy: 0.8001
Test Accuracy: 0.7995
Test Accuracy: 0.8237
Test Accuracy: 0.7942
Test Accuracy: 0.8109
Test Accuracy: 0.79
Test Accuracy: 0.8118
Test Accuracy: 0.7598
Test Accuracy: 0.8084
Test Accuracy:

In [23]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(28 * 28, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.net(x)

model = MLP()

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

epochs = 30
for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            preds = outputs.argmax(dim = 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}: Test Accuracy = {acc:.4f}")

Epoch 1: Test Accuracy = 0.8544
Epoch 2: Test Accuracy = 0.8632
Epoch 3: Test Accuracy = 0.8640
Epoch 4: Test Accuracy = 0.8671
Epoch 5: Test Accuracy = 0.8723
Epoch 6: Test Accuracy = 0.8711
Epoch 7: Test Accuracy = 0.8764
Epoch 8: Test Accuracy = 0.8779
Epoch 9: Test Accuracy = 0.8809
Epoch 10: Test Accuracy = 0.8797
Epoch 11: Test Accuracy = 0.8855
Epoch 12: Test Accuracy = 0.8843
Epoch 13: Test Accuracy = 0.8838
Epoch 14: Test Accuracy = 0.8842
Epoch 15: Test Accuracy = 0.8831
Epoch 16: Test Accuracy = 0.8812
Epoch 17: Test Accuracy = 0.8883
Epoch 18: Test Accuracy = 0.8840
Epoch 19: Test Accuracy = 0.8798
Epoch 20: Test Accuracy = 0.8846
Epoch 21: Test Accuracy = 0.8796
Epoch 22: Test Accuracy = 0.8780
Epoch 23: Test Accuracy = 0.8796
Epoch 24: Test Accuracy = 0.8817
Epoch 25: Test Accuracy = 0.8869
Epoch 26: Test Accuracy = 0.8869
Epoch 27: Test Accuracy = 0.8841
Epoch 28: Test Accuracy = 0.8841
Epoch 29: Test Accuracy = 0.8849
Epoch 30: Test Accuracy = 0.8840


In [24]:
model_scripted = torch.jit.script(model)


model_scripted.save("fashion.pt")